# Validación de marts: reclasificación de superficie, regla de exclusión y dimensiones

**Objetivo:** documentar las decisiones tomadas al diseñar `models/marts/` — la reclasificación
de superficie centinela/habitación, la regla de exclusión de `fct_arriendos` (4 criterios), la
investigación que llevó a usar P99 de `precio_clp_por_m2` por fuente, y la validación de
`dim_comuna` / `dim_calendario`.

Continúa a `notebooks/03_validate_staging.ipynb` (perfilado inicial de valores centinela). Este
notebook es solo de validación/diagnóstico: no transforma datos ni reemplaza la lógica de
negocio, que vive en `models/staging/` y `models/marts/`. Consulta directamente `staging` y
`marts` en BigQuery.

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google.cloud import bigquery

load_dotenv("../.env")

cred_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
if cred_path and not os.path.isabs(cred_path):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str((Path("..") / cred_path).resolve())

PROJECT = os.getenv("GCP_PROJECT_ID")
DATASET_STAGING = os.getenv("BQ_DATASET_STAGING")
DATASET_MARTS = os.getenv("BQ_DATASET_MARTS")

bq = bigquery.Client(project=PROJECT)


def query(sql: str) -> pd.DataFrame:
    return bq.query(sql).to_dataframe()


CENTINELA = (1, 2, 250, 400, 650, 999, 1200)
HABITACION = (5, 8, 10, 12)

## 1. Reclasificación de superficie: por qué 1/2 quedan centinela y 5/8/10/12 pasan a "habitación"

`03_validate_staging.ipynb` investigó originalmente 1, 2 y 5 como un solo grupo. Al revisar el
rango completo 1-20 m² se encontró que **1, 2, 5, 8, 10, 12** comparten la misma firma
estadística (picos exactos sin ningún valor vecino decimal), separada de la familia
250/400/650/999/1200. La decisión final del negocio fue tratar solo 1 y 2 como centinela, y
5/8/10/12 como "habitación" (superficie real, pero de un espacio no residencial estándar).

In [2]:
for tabla, col in [("stg_toctoc", "superficie_m2_original"), ("stg_portal_inmobiliario", "superficie_m2_original")]:
    df = query(f'''
        SELECT superficie_categoria, COUNT(*) AS n
        FROM `{PROJECT}.{DATASET_STAGING}.{tabla}`
        GROUP BY superficie_categoria
        ORDER BY superficie_categoria
    ''')
    print(f"--- {tabla} ---")
    display(df)

--- stg_toctoc ---


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,superficie_categoria,n
0,centinela,996
1,habitacion,475
2,valida,122731


--- stg_portal_inmobiliario ---


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,superficie_categoria,n
0,centinela,939
1,habitacion,490
2,valida,120666


## 2. Regla de exclusión de `fct_arriendos` (3 criterios)

In [3]:
for tabla in ["stg_toctoc", "stg_portal_inmobiliario"]:
    df = query(f'''
        SELECT
          COUNT(*) AS total,
          COUNTIF(superficie_categoria = 'centinela') AS excluido_centinela,
          COUNTIF(superficie_categoria = 'habitacion') AS excluido_habitacion,
          COUNTIF(superficie_categoria != 'valida') AS excluido_criterios_1_a_2
        FROM `{PROJECT}.{DATASET_STAGING}.{tabla}`
    ''')
    print(f"--- {tabla} ---")
    display(df)


--- stg_toctoc ---


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,total,excluido_centinela,excluido_habitacion,excluido_criterios_1_a_2
0,124202,996,475,1471


--- stg_portal_inmobiliario ---


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,total,excluido_centinela,excluido_habitacion,excluido_criterios_1_a_2
0,122095,939,490,1429


## 3. Investigación del tercer criterio: P99 de `precio_clp_por_m2` por fuente

Antes de fijar la regla se revisó `precio_clp_por_m2` **por comuna y superficie exactas** en
Vitacura 22-24 m², que mostró una discontinuidad clara entre lo "normal" y los outliers.

In [4]:
df = query(f'''
    SELECT fuente, id, precio_clp, superficie_m2,
           ROUND(precio_clp / superficie_m2, 0) AS precio_clp_por_m2
    FROM `{PROJECT}.{DATASET_STAGING}.stg_arriendos`
    WHERE comuna = 'Vitacura' AND superficie_m2 BETWEEN 22 AND 24
    ORDER BY precio_clp
''')
print(f"{len(df)} avisos. Rango sin los 3 outliers: "
      f"{df['precio_clp'].iloc[:-3].min():,.0f} - {df['precio_clp'].iloc[:-3].max():,.0f} CLP")
display(df)

29 avisos. Rango sin los 3 outliers: 720,000 - 1,007,287 CLP


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,fuente,id,precio_clp,superficie_m2,precio_clp_por_m2
0,toctoc,TT-2023-07-002557,720000,23.8,30252.0
1,toctoc,TT-2023-05-002145,723897,22.3,32462.0
2,toctoc,TT-2023-03-003144,790000,23.9,33054.0
3,toctoc,TT-2023-05-005046,795000,22.8,34868.0
4,toctoc,TT-2023-11-000307,805000,24.0,33542.0
5,portalinmobiliario,PI-2023-12-000773,820000,22.9,35808.0
6,portalinmobiliario,PI-2023-06-003259,820000,22.7,36123.0
7,portalinmobiliario,PI-2023-09-004724,830000,23.7,35021.0
8,portalinmobiliario,PI-2023-06-000023,845000,22.8,37061.0
9,portalinmobiliario,PI-2024-08-000002,860000,23.2,37069.0


No hay gradiente entre ~1M y los outliers (2,5M y ~9M) — salto directo, sin valores
intermedios. Esto confirmó que un corte por percentil (no un umbral fijo en CLP/m², y no IQR)
era razonable, calculado **por fuente** ya que Toctoc y Portal tienen distribuciones de precio
distintas.

In [5]:
for fuente in ["toctoc", "portalinmobiliario"]:
    df = query(f'''
        WITH poblacion AS (
          SELECT precio_clp / superficie_m2 AS precio_clp_por_m2
          FROM `{PROJECT}.{DATASET_STAGING}.stg_arriendos`
          WHERE fuente = '{fuente}'
            AND superficie_categoria = 'valida'
        )
        SELECT
          APPROX_QUANTILES(precio_clp_por_m2, 100)[OFFSET(95)] AS p95,
          APPROX_QUANTILES(precio_clp_por_m2, 100)[OFFSET(99)] AS p99,
          MAX(precio_clp_por_m2) AS maximo,
          COUNT(*) AS n_poblacion
        FROM poblacion
    ''')
    print(fuente)
    display(df)


toctoc


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,p95,p99,maximo,n_poblacion
0,23897.058824,67679.127726,389746.738197,122731


portalinmobiliario


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,p95,p99,maximo,n_poblacion
0,23856.20915,84029.227557,397577.092511,120666


## 4. `dim_comuna`: validación

In [6]:
df = query(f'''
    SELECT COUNT(*) AS n_comunas,
           COUNT(DISTINCT comuna_id) AS n_ids_distintos,
           COUNT(DISTINCT region) AS n_regiones
    FROM `{PROJECT}.{DATASET_MARTS}.dim_comuna`
''')
display(df)

C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,n_comunas,n_ids_distintos,n_regiones
0,25,25,1


## 5. `dim_calendario`: validación de cobertura

In [7]:
df = query(f'''
    SELECT COUNT(*) AS n_dias, MIN(fecha) AS min_fecha, MAX(fecha) AS max_fecha
    FROM `{PROJECT}.{DATASET_MARTS}.dim_calendario`
''')
display(df)

C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,n_dias,min_fecha,max_fecha
0,1096,2022-01-01,2024-12-31


In [8]:
df = query(f'''
    SELECT COUNT(*) AS filas_sin_calendario
    FROM `{PROJECT}.{DATASET_MARTS}.fct_arriendos` f
    LEFT JOIN `{PROJECT}.{DATASET_MARTS}.dim_calendario` c ON f.fecha_scraping = c.fecha
    WHERE c.fecha IS NULL
''')
display(df)

C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,filas_sin_calendario
0,0


## 6. Validación final de `fct_arriendos`

In [9]:
df = query(f'''
    SELECT fuente, COUNT(*) AS n, ROUND(MAX(precio_clp_por_m2), 0) AS max_precio_por_m2
    FROM `{PROJECT}.{DATASET_MARTS}.fct_arriendos`
    GROUP BY fuente
''')
display(df)

C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,fuente,n,max_precio_por_m2
0,portalinmobiliario,119459,85259.0
1,toctoc,121503,67659.0


In [10]:
df = query(f'''
    SELECT id
    FROM `{PROJECT}.{DATASET_MARTS}.fct_arriendos`
    WHERE id IN ('TT-2023-12-001509','PI-2023-03-000850','TT-2023-01-003327')
''')
print("Casos anómalos de Vitacura que deberían quedar excluidos:", df["id"].tolist())

Casos anómalos de Vitacura que deberían quedar excluidos: []


C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [11]:
df = query(f'''
    SELECT
      (SELECT COUNT(*) FROM `{PROJECT}.{DATASET_STAGING}.stg_toctoc`) AS stg_toctoc,
      (SELECT COUNT(*) FROM `{PROJECT}.{DATASET_STAGING}.stg_portal_inmobiliario`) AS stg_portal,
      (SELECT COUNT(*) FROM `{PROJECT}.{DATASET_MARTS}.fct_arriendos`) AS fct_arriendos
''')
display(df)

C:\Claude\proyectos\pipeline-captacion-inmobiliaria\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,stg_toctoc,stg_portal,fct_arriendos
0,124202,122095,240962


## Resumen de validación y decisiones

### Hallazgos

- 1, 2, 5, 8, 10, 12 m² comparten la misma firma estadística (picos exactos sin vecino decimal),
  separada de 250/400/650/999/1200.
- En Vitacura 22-24 m², el precio salta directo de ~1M a 2,5M y ~9M CLP, sin gradiente — no es
  un segmento premium continuo, son outliers aislados.

### Decisiones

- `superficie_categoria` reemplaza a `superficie_es_centinela`: `'centinela'` (1,2,250,400,650,
  999,1200) anula `superficie_m2`; `'habitacion'` (5,8,10,12) conserva `superficie_m2` real pero
  se excluye de `fct_arriendos` por ser una población distinta (no departamento estándar).
- `fct_arriendos` excluye 3 criterios: centinela, habitación, y el 1% superior de
  `precio_clp_por_m2` **por fuente** (no un umbral fijo en CLP, no IQR) — regla de
  representatividad analítica, no de corrección de datos erróneos.
- Ningún registro se elimina de `staging` — la exclusión ocurre únicamente en `fct_arriendos`.
- Modelo de marts final: `fct_arriendos` + `dim_comuna` + `dim_calendario`. Sin
  `fct_comuna_mensual` — las agregaciones se calculan en DAX dentro de Power BI.
- `dim_calendario` se genera en BigQuery (no depende de Power BI), cobertura 2022-01-01 a
  2024-12-31 para cubrir tanto `fecha_publicacion` como `fecha_scraping`.

### Validaciones

- Conteos de `superficie_categoria` en staging coinciden con lo esperado (996/475 Toctoc,
  939/490 Portal).
- `dim_comuna`: 25 comunas, sin duplicados de `comuna_id`.
- `dim_calendario`: 0 filas de `fct_arriendos` sin match por `fecha_scraping`.
- Los 3 casos anómalos de Vitacura (`TT-2023-12-001509`, `PI-2023-03-000850`,
  `TT-2023-01-003327`) confirmados fuera de `fct_arriendos`.
- `fct_arriendos` = staging válido menos el 1% superior de precio/m² por fuente, sin filas
  eliminadas de staging.
